In [ ]:
import sys; sys.path.insert(0, "..")
from src.pipeline.orchestrator import ARGUSPipeline
from src.models.stage2_economics import calculate_economic_impact
pipeline = ARGUSPipeline()
print("ARGUS pipeline loaded ✓")

In [ ]:
result = pipeline.run(lat_min=37, lat_max=40, lon_min=55, lon_max=60)
print(f"Run ID: {result.run_id}")
print(f"Detections: {len(result.detections)}")
print(f"Super-emitters: {result.n_super_emitters}")
print(f"Total flux: {result.total_flux_kg_hr:.1f} kg/hr")

In [ ]:
for det in result.detections[:3]:
    flux = det.get("flux_kg_hr", 0)
    econ = calculate_economic_impact(flux)
    print(econ.summary_line)
    for line in econ.detail_lines:
        print(" ", line)

In [ ]:
import matplotlib.pyplot as plt
import torch
from src.data.tropomi import TROPOMIIngester
from src.models.stage1_sat import preprocess_tropomi, mc_predict, load_model

ds   = TROPOMIIngester._mock(37, 40, 55, 60)
t    = preprocess_tropomi(ds)
mdl  = load_model()
out  = mc_predict(mdl, t)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(out.mask_mean.squeeze(), cmap="hot")
axes[0].set_title("Plume Probability (mean)")
axes[1].imshow(out.mask_variance.squeeze(), cmap="Blues")
axes[1].set_title("Epistemic Uncertainty (variance)")
plt.tight_layout()
plt.savefig("uncertainty_map.png", dpi=150)
plt.show()
print("Uncertainty map saved ✓")

In [ ]:
import requests
resp = requests.post("http://localhost:8000/api/v1/detect", json={
    "lat_min": 37, "lat_max": 40,
    "lon_min": 55, "lon_max": 60,
})
print(resp.json()["summary"])